In [1]:
import pandas as pd

df = pd.read_stata("patentvc_enhanced_4var.dta")
df.to_parquet("patentvc_enhanced_4var.parquet")

df = pd.read_parquet("patentvc_enhanced_4var.parquet")

In [2]:
df.head()

,patent_number,VC,grant_date,application_date
0,03930271,0.0,1976-01-06,1974-12-16
1,03930272,0.0,1976-01-06,1974-09-10
2,03930273,0.0,1976-01-06,1975-01-10
3,03930274,0.0,1976-01-06,1974-10-11
4,03930275,0.0,1976-01-06,1975-03-31


In [ ]:
import requests
import json
import pandas as pd

# Define headers per SEC guidelines
headers = {'User-Agent': 'UniversityResearch student@dartmouth.edu'}

def get_company_form_d(cik):
    # Standardize CIK to 10 digits with leading zeros
    cik_formatted = str(cik).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_formatted}.json"
    
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        recent_filings = data['filings']['recent']
        
        # Convert filings to DataFrame
        df_filings = pd.DataFrame(recent_filings)
        
        # Filter strictly for Form D (Notice of Exempt Offering of Securities)
        form_d_filings = df_filings[df_filings['form'] == 'D']
        return form_d_filings
    else:
        return None

# Example usage for a target company CIK
# form_d_data = get_company_form_d("0001234567")

In [10]:
import os
import duckdb
import pandas as pd

# 1. Define paths
base_dir = os.path.expanduser("~/Documents/projects/qss20_final/data")
dta_path = os.path.join(base_dir, "patentvc_enhanced_4var.dta")
uspto_dir = os.path.join(base_dir, "uspto")
db_path = os.path.join(base_dir, "uspto_research.db")

# 2. Load the Stata target dataset
print("Loading Stata file...")
df_stata = pd.read_stata(dta_path)
print("Stata file loaded! Columns found:", df_stata.columns.tolist())

# 3. Connect to DuckDB and register DataFrame
con = duckdb.connect(db_path)
con.register("df_stata_temp", df_stata)

# 4. Create target_patents table
con.execute("""
    CREATE OR REPLACE TABLE target_patents AS 
    SELECT 
        CAST(patent_number AS VARCHAR) AS patent_id, 
        application_date, 
        grant_date, 
        VC AS vc_backed
    FROM df_stata_temp;
""")

target_count = con.execute("SELECT COUNT(*) FROM target_patents;").fetchone()[0]
print(f"Loaded {target_count:,} target patents into 'target_patents'.")

# 5. Extract Citation Metrics (PVGPATDIS Zipped TSVs)
citation_zip = os.path.join(uspto_dir, "g_us_patent_citation.tsv.zip")
npl_zip = os.path.join(uspto_dir, "g_other_reference.tsv.zip")

if os.path.exists(citation_zip) and os.path.exists(npl_zip):
    print("Processing citation metrics from PVGPATDIS zip files...")
    con.execute(f"""
        CREATE OR REPLACE TABLE target_citation_metrics AS
        SELECT 
            p.patent_id,
            COUNT(DISTINCT c.citation_patent_id) AS backward_pat_citations,
            COUNT(DISTINCT n.other_reference_text) AS npl_citations,
            CAST(COUNT(DISTINCT n.other_reference_text) AS FLOAT) / 
                NULLIF(COUNT(DISTINCT c.citation_patent_id) + COUNT(DISTINCT n.other_reference_text), 0) AS npl_ratio
        FROM target_patents p
        LEFT JOIN read_csv('{citation_zip}', delim='\t', header=True, all_varchar=True, ignore_errors=True) c 
            ON p.patent_id = CAST(c.patent_id AS VARCHAR)
        LEFT JOIN read_csv('{npl_zip}', delim='\t', header=True, all_varchar=True, ignore_errors=True) n 
            ON p.patent_id = CAST(n.patent_id AS VARCHAR)
        GROUP BY p.patent_id;
    """)
    print("Created 'target_citation_metrics' table successfully.")

# 6. Extract Claim Stats from Stata Zip File
claims_dta_zip = os.path.join(uspto_dir, "patent_claims_stats.dta.zip")

if os.path.exists(claims_dta_zip):
    print("Processing claim statistics from patent_claims_stats.dta.zip...")
    df_claims = pd.read_stata(claims_dta_zip)
    con.register("df_claims_temp", df_claims)
    
    pat_col = [col for col in df_claims.columns if 'pat' in col.lower()][0]
    
    con.execute(f"""
        CREATE OR REPLACE TABLE target_claim_metrics AS
        SELECT 
            p.patent_id,
            c.* EXCLUDE ({pat_col})
        FROM target_patents p
        LEFT JOIN df_claims_temp c ON p.patent_id = CAST(c.{pat_col} AS VARCHAR);
    """)
    print("Created 'target_claim_metrics' table successfully.")

# 7. Extract Assignees & CPC Classifications (Zipped TSVs)
assignee_zip = os.path.join(uspto_dir, "g_assignee_disambiguated.tsv.zip")
cpc_zip = os.path.join(uspto_dir, "g_cpc_current.tsv.zip")

if os.path.exists(assignee_zip):
    print("Processing assignee disambiguation...")
    con.execute(f"""
        CREATE OR REPLACE TABLE target_assignee_metrics AS
        SELECT 
            p.patent_id,
            FIRST(a.disambig_assignee_organization) AS assignee_organization,
            FIRST(a.assignee_type) AS assignee_type
        FROM target_patents p
        LEFT JOIN read_csv('{assignee_zip}', delim='\t', header=True, all_varchar=True, ignore_errors=True) a 
            ON p.patent_id = CAST(a.patent_id AS VARCHAR)
        GROUP BY p.patent_id;
    """)

if os.path.exists(cpc_zip):
    print("Processing CPC classifications...")
    con.execute(f"""
        CREATE OR REPLACE TABLE target_cpc_metrics AS
        SELECT 
            p.patent_id,
            FIRST(c.cpc_section) AS main_cpc_section,
            FIRST(c.cpc_subclass) AS main_cpc_subclass
        FROM target_patents p
        LEFT JOIN read_csv('{cpc_zip}', delim='\t', header=True, all_varchar=True, ignore_errors=True) c 
            ON p.patent_id = CAST(c.patent_id AS VARCHAR)
        GROUP BY p.patent_id;
    """)

# 8. Merge Everything into Final Master Panel Table
print("Building master panel table...")
con.execute("""
    CREATE OR REPLACE TABLE master_patent_panel AS
    SELECT 
        p.*,
        c.backward_pat_citations,
        c.npl_citations,
        c.npl_ratio,
        cl.* EXCLUDE (patent_id),
        a.assignee_organization,
        a.assignee_type,
        cpc.main_cpc_section,
        cpc.main_cpc_subclass
    FROM target_patents p
    LEFT JOIN target_citation_metrics c ON p.patent_id = c.patent_id
    LEFT JOIN target_claim_metrics cl ON p.patent_id = cl.patent_id
    LEFT JOIN target_assignee_metrics a ON p.patent_id = a.patent_id
    LEFT JOIN target_cpc_metrics cpc ON p.patent_id = cpc.patent_id;
""")

# 9. Export to CSV
output_csv = os.path.join(base_dir, "cleaned_patent_panel.csv")
con.execute(f"COPY master_patent_panel TO '{output_csv}' (HEADER, DELIMITER ',');")
print(f"DONE! Master panel saved to {output_csv}")

Loading Stata file...
Stata file loaded! Columns found: ['patent_number', 'VC', 'grant_date', 'application_date']
Loaded 7,842,115 target patents into 'target_patents'.
Processing citation metrics from PVGPATDIS zip files...


InvalidInputException: Invalid Input Error: Error when sniffing file "/Users/test/Documents/projects/qss20_final/data/uspto/g_us_patent_citation.tsv.zip".
It was not possible to automatically detect the CSV parsing dialect
The search space used was:
Delimiter Candidates: '	'
Quote/Escape Candidates: ['(no quote)','(no escape)'],['"','(no escape)'],['"','"'],['"','''],['"','\'],[''','(no escape)'],[''','''],[''','"'],[''','\']
Comment Candidates: '\0', '#'
Encoding: utf-8
Possible fixes:
* Disable the parser's strict mode (strict_mode=false) to allow reading rows that do not comply with the CSV standard.
* Make sure you are using the correct file encoding. If not, set it (e.g., encoding = 'utf-16').
* Delimiter is set to '	'. Consider unsetting it.
* Set quote (e.g., quote='"')
* Set escape (e.g., escape='"')
* Set comment (e.g., comment='#')
* Set skip (skip=${n}) to skip ${n} lines at the top of the file
* Enable null padding (null_padding=true) to pad missing columns with NULL values

In [6]:
import os
import gzip
import zipfile
import pandas as pd

base_dir = os.path.expanduser("~/Documents/projects/qss20_final/data")
uspto_dir = os.path.join(base_dir, "uspto")

print("Checking USPTO directory:", uspto_dir)
if not os.path.exists(uspto_dir):
    print("Folder does not exist! Check path.")
else:
    files = [f for f in os.listdir(uspto_dir) if not f.startswith('.')]
    print(f"Found {len(files)} files in uspto folder:\n")
    
    for f in sorted(files):
        full_path = os.path.join(uspto_dir, f)
        print(f"File: {f}")
        try:
            if f.endswith('.zip'):
                with zipfile.ZipFile(full_path, 'r') as z:
                    first_file = z.namelist()[0]
                    with z.open(first_file) as file_obj:
                        df_sample = pd.read_csv(file_obj, sep=None, engine='python', nrows=2)
                        print(f"   (Inside zip: {first_file})")
                        print(f"   Columns: {df_sample.columns.tolist()}")
            elif f.endswith('.gz'):
                with gzip.open(full_path, 'rt') as gz:
                    df_sample = pd.read_csv(gz, sep=None, engine='python', nrows=2)
                    print(f"   Columns: {df_sample.columns.tolist()}")
            else:
                df_sample = pd.read_csv(full_path, sep=None, engine='python', nrows=2)
                print(f"   Columns: {df_sample.columns.tolist()}")
        except Exception as e:
            print(f"   Could not read header: {e}")
        print("-" * 60)

Checking USPTO directory: /Users/test/Documents/projects/qss20_final/data/uspto
Found 7 files in uspto folder:

File: assignee.csv.zip
   (Inside zip: assignee.csv)
   Columns: ['rf_id', 'ee_name', 'ee_address_1', 'ee_address_2', 'ee_city', 'ee_state', 'ee_postcode', 'ee_country']
------------------------------------------------------------
File: assignment.csv.zip
   (Inside zip: assignment.csv)
   Columns: ['rf_id', 'file_id', 'cname', 'caddress_1', 'caddress_2', 'caddress_3', 'caddress_4', 'reel_no', 'frame_no', 'convey_text', 'record_dt', 'last_update_dt', 'page_count', 'purge_in']
------------------------------------------------------------
File: g_assignee_disambiguated.tsv.zip
   (Inside zip: g_assignee_disambiguated.tsv)
   Columns: ['patent_id', 'assignee_sequence', 'assignee_id', 'disambig_assignee_individual_name_first', 'disambig_assignee_individual_name_last', 'disambig_assignee_organization', 'assignee_type', 'location_id']
------------------------------------------------

In [ ]:
import os
import zipfile
import duckdb
import pandas as pd

# 1. Paths
base_dir = os.path.expanduser("~/Documents/projects/qss20_final/data")
dta_path = os.path.join(base_dir, "patentvc_enhanced_4var.dta")
uspto_dir = os.path.join(base_dir, "uspto")
db_path = os.path.join(base_dir, "uspto_research.db")

# 2. Connect & Setup Target Table
print("Loading Stata file...")
df_stata = pd.read_stata(dta_path)
con = duckdb.connect(db_path)
con.register("df_stata_temp", df_stata)

con.execute("""
    CREATE OR REPLACE TABLE target_patents AS 
    SELECT 
        CAST(patent_number AS VARCHAR) AS patent_id, 
        application_date, 
        grant_date, 
        VC AS vc_backed
    FROM df_stata_temp;
""")
print(f"Loaded {con.execute('SELECT COUNT(*) FROM target_patents;').fetchone()[0]:,} target patents.")


# Helper function to unzip one file, query it with DuckDB, and delete unzipped file
def process_single_zip(zip_name, query_func):
    zip_path = os.path.join(uspto_dir, zip_name)
    if not os.path.exists(zip_path):
        print(f"Skipping {zip_name} (file not found)")
        return
    
    print(f"\n[Space-Safe] Unzipping {zip_name}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        extracted_file = zip_ref.namelist()[0]
        zip_ref.extractall(uspto_dir)
        extracted_path = os.path.join(uspto_dir, extracted_file)
    
    print(f"Processing {extracted_file} in DuckDB...")
    query_func(extracted_path)
    
    # Immediately remove unzipped file to free disk space
    if os.path.exists(extracted_path):
        os.remove(extracted_path)
        print(f"Cleaned up {extracted_file} (Disk space restored!)")


# 3. Process Citations with Explicit VARCHAR Types
def run_citations(extracted_path):
    if "citation" in extracted_path:
        con.execute(f"""
            CREATE OR REPLACE TABLE raw_citations AS 
            SELECT CAST(patent_id AS VARCHAR) AS patent_id, CAST(citation_patent_id AS VARCHAR) AS citation_patent_id 
            FROM read_csv_auto('{extracted_path}', types={{'patent_id': 'VARCHAR', 'citation_patent_id': 'VARCHAR'}});
        """)
    elif "reference" in extracted_path:
        con.execute(f"""
            CREATE OR REPLACE TABLE raw_npl AS 
            SELECT CAST(patent_id AS VARCHAR) AS patent_id, other_reference_text 
            FROM read_csv_auto('{extracted_path}', types={{'patent_id': 'VARCHAR'}});
        """)

process_single_zip("g_us_patent_citation.tsv.zip", run_citations)
process_single_zip("g_other_reference.tsv.zip", run_citations)

if con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'raw_citations'").fetchone()[0] > 0:
    print("Building target_citation_metrics table...")
    con.execute("""
        CREATE OR REPLACE TABLE target_citation_metrics AS
        SELECT 
            p.patent_id,
            COUNT(DISTINCT c.citation_patent_id) AS backward_pat_citations,
            COUNT(DISTINCT n.other_reference_text) AS npl_citations,
            CAST(COUNT(DISTINCT n.other_reference_text) AS FLOAT) / 
                NULLIF(COUNT(DISTINCT c.citation_patent_id) + COUNT(DISTINCT n.other_reference_text), 0) AS npl_ratio
        FROM target_patents p
        LEFT JOIN raw_citations c ON p.patent_id = c.patent_id
        LEFT JOIN raw_npl n ON p.patent_id = n.patent_id
        GROUP BY p.patent_id;
    """)
    con.execute("DROP TABLE IF EXISTS raw_citations; DROP TABLE IF EXISTS raw_npl;")


# 4. Process Claims (Stata zip)
claims_dta_zip = os.path.join(uspto_dir, "patent_claims_stats.dta.zip")
if os.path.exists(claims_dta_zip):
    print("\nProcessing claim statistics...")
    df_claims = pd.read_stata(claims_dta_zip)
    con.register("df_claims_temp", df_claims)
    pat_col = [col for col in df_claims.columns if 'pat' in col.lower()][0]
    
    con.execute(f"""
        CREATE OR REPLACE TABLE target_claim_metrics AS
        SELECT 
            p.patent_id,
            c.* EXCLUDE ({pat_col})
        FROM target_patents p
        LEFT JOIN df_claims_temp c ON p.patent_id = CAST(c.{pat_col} AS VARCHAR);
    """)


# 5. Process Assignees
def run_assignees(extracted_path):
    con.execute(f"""
        CREATE OR REPLACE TABLE target_assignee_metrics AS
        SELECT 
            p.patent_id,
            FIRST(a.disambig_assignee_organization) AS assignee_organization,
            FIRST(a.assignee_type) AS assignee_type
        FROM target_patents p
        LEFT JOIN read_csv_auto('{extracted_path}', types={{'patent_id': 'VARCHAR'}}) a ON p.patent_id = a.patent_id
        GROUP BY p.patent_id;
    """)

process_single_zip("g_assignee_disambiguated.tsv.zip", run_assignees)


# 6. Process CPC
def run_cpc(extracted_path):
    con.execute(f"""
        CREATE OR REPLACE TABLE target_cpc_metrics AS
        SELECT 
            p.patent_id,
            FIRST(c.cpc_section) AS main_cpc_section,
            FIRST(c.cpc_subclass) AS main_cpc_subclass
        FROM target_patents p
        LEFT JOIN read_csv_auto('{extracted_path}', types={{'patent_id': 'VARCHAR'}}) c ON p.patent_id = c.patent_id
        GROUP BY p.patent_id;
    """)

process_single_zip("g_cpc_current.tsv.zip", run_cpc)


# 7. Merge Master Panel
print("\nBuilding master panel table...")
con.execute("""
    CREATE OR REPLACE TABLE master_patent_panel AS
    SELECT 
        p.*,
        c.backward_pat_citations,
        c.npl_citations,
        c.npl_ratio,
        cl.* EXCLUDE (patent_id),
        a.assignee_organization,
        a.assignee_type,
        cpc.main_cpc_section,
        cpc.main_cpc_subclass
    FROM target_patents p
    LEFT JOIN target_citation_metrics c ON p.patent_id = c.patent_id
    LEFT JOIN target_claim_metrics cl ON p.patent_id = cl.patent_id
    LEFT JOIN target_assignee_metrics a ON p.patent_id = a.patent_id
    LEFT JOIN target_cpc_metrics cpc ON p.patent_id = cpc.patent_id;
""")

output_csv = os.path.join(base_dir, "cleaned_patent_panel.csv")
con.execute(f"COPY master_patent_panel TO '{output_csv}' (HEADER, DELIMITER ',');")
print(f"\n🎉 SUCCESS! Master panel saved to {output_csv}")

Loading Stata file...
Loaded 7,842,115 target patents.

[Space-Safe] Unzipping g_us_patent_citation.tsv.zip...
Processing g_us_patent_citation.tsv in DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cleaned up g_us_patent_citation.tsv (Disk space restored!)

[Space-Safe] Unzipping g_other_reference.tsv.zip...
